<a href="https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("FlyRank-ML")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [6]:
import duckdb

con = duckdb.connect()

print("DuckDB connected:", con is not None)

DuckDB connected: True


In [7]:
con = duckdb.connect()

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [8]:
feature_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("March rows:", len(feature_frame))

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 9841378


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

Prioritize content for review when it has meaningful search visibility but weak click engagement, especially when the page already ranks within a relatively strong search position.

The baseline score combines search impressions, CTR, and search position. Higher scores indicate higher review priority.

### Reason codes

- `HIGH_IMPRESSIONS_LOW_CTR` — high search visibility combined with weak CTR.
- `GOOD_POSITION_LOW_CTR` — relatively strong search position but weak CTR, suggesting a potential CTR-improvement opportunity.
- `HIGH_VISIBILITY` — substantial search visibility, but the stronger CTR/position conditions for the other reason codes are not met.
- `REVIEW` — lower-priority candidate retained in the ranked queue for comparison.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


The baseline score ranks content by review priority using search visibility, CTR opportunity, and search-position opportunity.

The queue assigns one reason code and one action label to each client-content pair, then sorts all rows by the baseline score in descending order.

The ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [17]:
# Thresholds are based on the signal audit in ML-06
# Good search position: top 10
# Low CTR: below 0.1%
# High impressions: more than 100 impressions

import numpy as np
import pandas as pd
import os


# ==================================================
# 1. Aggregate daily observations to content level
# ==================================================
# The warehouse data is daily.
# The baseline queue must contain one row per
# client-content pair.

baseline_df = (
    feature_frame
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_pageviews=("ga4_pageviews", "sum"),
        ga4_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)


# ==================================================
# 2. Calculate CTR
# ==================================================

baseline_df["ctr"] = (
    baseline_df["gsc_clicks"]
    / baseline_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)


# ==================================================
# 3. Baseline score
# ==================================================
# Higher score = higher review priority.
#
# Components:
# 1. Search visibility
# 2. CTR opportunity
# 3. Ranking opportunity

baseline_df["visibility_score"] = (
    baseline_df["gsc_impressions"].rank(pct=True)
)

baseline_df["ctr_opportunity"] = (
    1 - baseline_df["ctr"].rank(pct=True)
)


# Lower position = better ranking.
# Only positive/valid positions are used.

position_rank = baseline_df["gsc_avg_position"].where(
    baseline_df["gsc_avg_position"] > 0
)

baseline_df["position_opportunity"] = (
    1 - position_rank.rank(pct=True)
).fillna(0)


baseline_df["baseline_score"] = (
    baseline_df["visibility_score"]
    + baseline_df["ctr_opportunity"]
    + baseline_df["position_opportunity"]
) / 3


# ==================================================
# 4. ONE reason code per row
# ==================================================

baseline_df["reason_code"] = "REVIEW"


# High visibility + very low CTR
baseline_df.loc[
    (baseline_df["gsc_impressions"] > 100)
    & (baseline_df["ctr"] < 0.001),
    "reason_code"
] = "HIGH_IMPRESSIONS_LOW_CTR"


# Good position + very low CTR
baseline_df.loc[
    (baseline_df["gsc_avg_position"] <= 10)
    & (baseline_df["ctr"] < 0.001)
    & (baseline_df["reason_code"] == "REVIEW"),
    "reason_code"
] = "GOOD_POSITION_LOW_CTR"


# High visibility without the stronger CTR/position condition
baseline_df.loc[
    (baseline_df["gsc_impressions"] > 100)
    & (baseline_df["reason_code"] == "REVIEW"),
    "reason_code"
] = "HIGH_VISIBILITY"


# ==================================================
# 5. Action label
# ==================================================

baseline_df["action"] = "NO_ACTION"


baseline_df.loc[
    baseline_df["reason_code"].isin([
        "HIGH_IMPRESSIONS_LOW_CTR",
        "GOOD_POSITION_LOW_CTR"
    ]),
    "action"
] = "REVIEW_CTR"


baseline_df.loc[
    baseline_df["reason_code"] == "HIGH_VISIBILITY",
    "action"
] = "REVIEW"


# ==================================================
# 6. Build ranked queue
# ==================================================

baseline_df = (
    baseline_df
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

baseline_df["rank"] = baseline_df.index + 1


# ==================================================
# 7. Save ranked queue
# ==================================================

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_df.to_csv(
    output_path,
    index=False
)

print(f"Saved ranked queue to: {output_path}")


# ==================================================
# 8. Verification
# ==================================================

print("\nRows:", len(baseline_df))

print(
    "Unique client-content pairs:",
    baseline_df[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)

print("\nReason codes:")
print(baseline_df["reason_code"].value_counts())


# ==================================================
# 9. Top-20 ranked queue
# ==================================================

baseline_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20)

Saved ranked queue to: work/outputs/baseline_action_score.csv

Rows: 331437
Unique client-content pairs: 331437

Reason codes:
reason_code
REVIEW                      189924
HIGH_VISIBILITY              53977
HIGH_IMPRESSIONS_LOW_CTR     47255
GOOD_POSITION_LOW_CTR        40281
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,baseline_score,reason_code,action
0,1,client_e547b89c05043229,content_83167156f76e33e5,6827,0,0.0,1.136714,0.854320,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
1,2,client_a80fca3f171ed1de,content_fa17add7836d36c3,12588,0,0.0,1.902457,0.853611,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
2,3,client_73cda7b4e4f265ea,content_d397987113cb84a0,9887,0,0.0,1.953104,0.851407,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
3,4,client_62f4a7e64f5e0096,content_db01d94616cdb80d,4202,0,0.0,0.911944,0.849474,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
4,5,client_e547b89c05043229,content_ecc27d32010b18e0,3900,0,0.0,0.553854,0.849249,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
5,6,client_20259bd6705d81d4,content_48353a0ccb387d98,3795,0,0.0,0.844471,0.848060,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
6,7,client_73cda7b4e4f265ea,content_a100cf3244aa42db,3191,0,0.0,0.565396,0.845925,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
7,8,client_73cda7b4e4f265ea,content_1c3cadd8a8560665,2975,0,0.0,0.606774,0.844623,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
8,9,client_73cda7b4e4f265ea,content_858b2db26b9f62c4,2934,0,0.0,0.508723,0.844476,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR
9,10,client_73cda7b4e4f265ea,content_787b5229dfb9f4f6,3537,0,0.0,1.155720,0.844247,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR


### Result

The baseline produces a ranked queue for all client-content pairs in the March 2026 decision window. Each row contains a baseline score, one reason code, and an action label.



## 3. Top-20 review

The top 20 items are reviewed individually to check whether the baseline ranking is reasonable.

For each item, the review records the assigned action, reason code, confidence note, and a condition that could make the recommendation wrong. This is a manual sanity check rather than a new scoring method.

In [18]:
# ==================================================
# Top-20 manual review
# ==================================================

top20 = baseline_df.head(20).copy()


def make_confidence_note(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR":
        return (
            "High confidence: strong visibility with zero/very low CTR "
            "supports a CTR review."
        )

    elif row["reason_code"] == "GOOD_POSITION_LOW_CTR":
        return (
            "Moderate-high confidence: relatively strong position with "
            "weak CTR suggests a potential CTR opportunity."
        )

    elif row["reason_code"] == "HIGH_VISIBILITY":
        return (
            "Moderate confidence: high visibility supports review, "
            "but the CTR/position evidence is weaker."
        )

    return (
        "Lower confidence: retained in the ranked queue without "
        "a strong specific trigger."
    )


def make_wrong_condition(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR":
        return (
            "It could be wrong if the low CTR is expected for the query "
            "intent, SERP layout, or if the impression data is atypical."
        )

    elif row["reason_code"] == "GOOD_POSITION_LOW_CTR":
        return (
            "It could be wrong if the query intent does not normally "
            "generate clicks at this position or if CTR is naturally low."
        )

    elif row["reason_code"] == "HIGH_VISIBILITY":
        return (
            "It could be wrong if high impressions do not represent "
            "a meaningful content opportunity."
        )

    return (
        "It could be wrong if the available search signals do not "
        "represent a genuine content improvement opportunity."
    )


top20["confidence_note"] = top20.apply(
    make_confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    make_wrong_condition,
    axis=1
)


top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "baseline_score",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]


top20_review

,rank,client_hash_id,content_hash_id,action,reason_code,baseline_score,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,confidence_note,what_would_make_it_wrong
0,1,client_e547b89c05043229,content_83167156f76e33e5,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.854320,6827,0,0.0,1.136714,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
1,2,client_a80fca3f171ed1de,content_fa17add7836d36c3,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.853611,12588,0,0.0,1.902457,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
2,3,client_73cda7b4e4f265ea,content_d397987113cb84a0,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.851407,9887,0,0.0,1.953104,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
3,4,client_62f4a7e64f5e0096,content_db01d94616cdb80d,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.849474,4202,0,0.0,0.911944,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
4,5,client_e547b89c05043229,content_ecc27d32010b18e0,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.849249,3900,0,0.0,0.553854,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
5,6,client_20259bd6705d81d4,content_48353a0ccb387d98,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.848060,3795,0,0.0,0.844471,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
6,7,client_73cda7b4e4f265ea,content_a100cf3244aa42db,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.845925,3191,0,0.0,0.565396,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
7,8,client_73cda7b4e4f265ea,content_1c3cadd8a8560665,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.844623,2975,0,0.0,0.606774,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
8,9,client_73cda7b4e4f265ea,content_858b2db26b9f62c4,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.844476,2934,0,0.0,0.508723,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...
9,10,client_73cda7b4e4f265ea,content_787b5229dfb9f4f6,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,0.844247,3537,0,0.0,1.155720,High confidence: strong visibility with zero/v...,It could be wrong if the low CTR is expected f...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


---



The top-ranked queue is reviewed for weak picks that may look suspicious despite having a high baseline score.

The baseline uses only March reporting-period signals available at the decision moment. No future-window outcomes, product flags, or label-derived features are used in the scoring rule.

In [21]:
# ==================================================
# 4. Weak picks + leakage check
# ==================================================


# ==================================================
# 1. Identify potentially weak picks
# ==================================================
# A pick is considered potentially weak when it has
# high visibility but does not have strong supporting
# evidence from CTR or search position.
#
# Important:
# Zero clicks alone is NOT treated as a weak pick,
# because low CTR is intentionally part of the baseline rule.

weak_picks = top20_review[
    (
        (top20_review["reason_code"] == "HIGH_VISIBILITY")
        & (top20_review["gsc_avg_position"] > 20)
    )
    |
    (
        top20_review["reason_code"] == "REVIEW"
    )
].copy()


print("Potential weak picks:")

if len(weak_picks) == 0:
    print(
        "None among the top 20. "
        "The top-ranked candidates have a clear "
        "high-impressions/low-CTR signal."
    )
else:
    print(
        weak_picks[
            [
                "rank",
                "client_hash_id",
                "content_hash_id",
                "gsc_impressions",
                "gsc_clicks",
                "ctr",
                "gsc_avg_position",
                "reason_code",
                "action"
            ]
        ]
    )


# ==================================================
# 2. Leakage check
# ==================================================
# Check that no future-window, label-derived,
# deliberately leaked, or product-flag columns
# are present in the baseline dataframe.

baseline_columns = set(baseline_df.columns)

forbidden_terms = [
    "april",
    "may",
    "june",
    "future",
    "label",
    "leaked",
    "product_flag"
]

leakage_columns = [
    col
    for col in baseline_columns
    if any(
        term in col.lower()
        for term in forbidden_terms
    )
]


print("\nPotential leakage-related columns:")
print(leakage_columns)


# ==================================================
# 3. Confirm scoring inputs
# ==================================================
# These are the signals actually used to construct
# the baseline score.

score_inputs = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("\nScore inputs:")
print(score_inputs)


# ==================================================
# 4. Final leakage verdict
# ==================================================

print("\nLeakage check:")

if len(leakage_columns) == 0:
    print(
        "PASS — no future-window, label-derived, "
        "or product-flag columns are present."
    )
else:
    print(
        "REVIEW — potentially unsafe columns detected:",
        leakage_columns
    )

Potential weak picks:
None among the top 20. The top-ranked candidates have a clear high-impressions/low-CTR signal.

Potential leakage-related columns:
[]

Score inputs:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Leakage check:
PASS — no future-window, label-derived, or product-flag columns are present.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.